In [ ]:
import sys
import os

sys.path.append(os.path.abspath("../../"))


In [ ]:
!ls

In [ ]:
from __future__ import annotations

from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, RemoveMessage

from src.agent.graph import LangGraph
from dotenv import load_dotenv


In [ ]:
load_dotenv()


In [ ]:
memory = MemorySaver()
langgraph = LangGraph()
app = langgraph._build_graph(checkpointer=memory)

thread_id = "test_user_01"
config = {"configurable": {"thread_id": thread_id}}

In [ ]:
# # LLM 초기화 및 함수 호출을 통한 구조화된 출력 생성
# from langchain.chat_models import init_chat_model
# from langchain_openai import OpenAIEmbeddings, ChatOpenAI
# from src.AGENT.states import State, RouterDecision
# from langchain_core.prompts import ChatPromptTemplate

# llm = init_chat_model(
#             "solar-pro3",
#             model_provider="upstage",
#         )

# structured_llm = llm.with_structured_output(RouterDecision)

# # 시스템 메시지와 사용자 질문을 포함한 프롬프트 템플릿 생성
# ROUTER_SYSTEM_PROMPT = """
# 당신은 질문 분석가이자 사용자의 질문을 rag, web, weather, direct로 라우팅하는 전문가입니다. 반드시 단계적으로 사고해 주십시오. 영어로 대답시 소문자로 대답하세요.

# ** 다음과 같은 순서로 단계별로 질문을 분석하세요(CoT):
# 1. 질문을 분석합니다.
# - "검색", "추천" 과 같은 요청은 direct 가 아닙니다. 간단한 인사나, 여태까지 나눴던 대화의 요약과 같은 요청이 direct 요청입니다. 질문이 direct면, 밑에 단계를 건너뜁니다.
# 2. 질문을 세분화 합니다. 정보의 빈틈을 보완하기 위한 구체적이고 명확한 질문만 생성합니다. 질문은 명사로 나눠줍니다. 겹치는 질문은 생성하지 않습니다. 예를 들어, 깨끗한 부산의 숙소와 깨끗한 부산의 호텔은 같은 의미입니다. 하나만 고르시오.
# - 예시1. "부산의 내일 강수량과 바다마루 전복죽이라는 음식점의 운영시간과 부산의 숙박을 추천해줘" 라는 질문은 "부산의 내일 강수량", "바다마루 전복죽의 운영시간", "부산의 숙박" 이라는 3가지의 주제로 나눌 수 있습니다.
# - 예시2. "8월에 열리는 부산의 축제와 해운대의 호텔을 추천해줘." 라는 질문은 "8월에 열리는 부산 축제" 와 "해운대 호텔" 이라는 두가지 주제로 나눌 수 있습니다.
# 3. 세분화한 질문들을 각 rag, web, weather로 나눠 줍니다.
# - weather는 날씨와 관련된 정보들을 검색할 때 사용합니다. 날씨, 강수량, 일몰 시간, 일출 시간과 관련된 질문들이라면 weather를 사용하세요. 내일 날씨, 다음주 강수량과 같은 다른 날짜의 날씨 질문도 weather에게 물어봅니다.
# - rag는 음식점, 카페, 문화예술, 액티비티(체험), 쇼핑, 관광지, 축제, 숙박업소 (호텔, 모텔, 게스트하우스, 숙소 등등 포함) 를 포함합니다. 이러한 주제에 관한 질문들이라면 rag를 사용하세요.
# - web은 rag에 없는 정보들을 검색할 때 사용합니다. 예를 들어 식당이나 업소의 운영 시간, 가장 빠른 기차 시간 같은 최신성 정보가 필요한 경우 web을 사용합니다.
# 4. 각 세분화한 질문들을 다음과 같은 형태로 담아 반환해줍니다. 이때 질문 리스트들을 정리하여 뽑아내기 쉽도록 쿼리를 정리해주세요.

# 반드시 아래 JSON 형식으로만 출력하세요.
# 다른 문장, 설명, 마크다운, 특수 기호를 절대 포함하지 마세요.
# None 대신 null을 사용하세요.

# 출력형식:
# {{
#   "rag_queries": rag에 나눠진 질문 리스트. 만약 없으면 null 값을 반환합니다.,
#   "web_queries": web에 나눠진 질문 리스트. 만약 없으면 null 값을 반환합니다.,
#   "weather_queries": weather에 나눠진 질문 리스트. 만약 없으면 null 값을 반환합니다.,
#   "direct": direct로 넘겨야 할 경우 사용자의 질문을 넣어줍니다. 만약 direct 로 넘어가지 않는다면 null 값을 반환합니다.
#   "reason": 짧은 이유 설명.
# }}
# """

# gpt_prompt = """
# 당신은 사용자의 질문을 rag, web, weather, direct 중 하나 이상으로 분류하는 라우터입니다.

# 규칙:
# - 단순 인사, 대화 요약, 잡담은 direct
# - 날씨, 강수량, 일출/일몰 → weather
# - 음식점, 카페, 관광지, 숙박, 축제, 쇼핑, 체험 → rag
# - 운영시간, 교통시간, 최신 정보 → web

# 질문에 여러 주제가 있으면 각각 나눠 분류하세요.
# 쿼리는 짧은 명사 형태로 작성하세요.

# 반드시 JSON만 출력하세요.
# None 대신 null 사용.

# 출력 형식:
# {
#   "rag_queries": [],
#   "web_queries": [],
#   "weather_queries": [],
#   "direct": null,
#   "reason": "짧은 한 줄"
# }
# """

# gemini_PROMPT = """
# 당신은 사용자 질문을 의도에 따라 [rag, web, weather, direct]로 분류하고 핵심 쿼리를 추출하는 Router입니다.
# 생각 과정을 출력하지 말고, 입력된 질문을 분석하여 즉시 아래 JSON 포맷으로만 응답하세요.

# **분류 및 쿼리 추출 규칙:**
# 1. **direct:** 검색/추천 요청이 아닌 경우(인사, 대화 요약, 농담 등). 이 경우 다른 쿼리는 null.
# 2. **weather:** 날씨, 기온, 강수량, 일출/일몰 등 기상 정보.
# 3. **rag:** 장소, 숙박(호텔/숙소), 맛집, 관광지, 쇼핑, 축제 등 '장소/대상' 자체에 대한 정보.
# 4. **web:** rag에 없는 실시간 정보(운영 시간, 기차표 시간/예매, 최신 뉴스) 등.

# **작성 지침:**
# - 복합 질문은 주제별로 명사형 쿼리로 분리하세요. (중복 제거)
# - 설명, 마크다운(```json)을 절대 포함하지 마세요.
# - 해당사항이 없는 항목은 null을 반환하세요.

# **출력 형식:**
# {{
#   "rag_queries": ["쿼리1", "쿼리2"] 또는 null,
#   "web_queries": ["쿼리1", "쿼리2"] 또는 null,
#   "weather_queries": ["쿼리1", "쿼리2"] 또는 null,
#   "direct": "사용자 질문 원문" 또는 null,
#   "reason": "분류 이유 간략 요약"
# }}
# """


In [ ]:
# # # add system prompt
# # question = "부산의 내일 강수량과 바다마루 전복죽이라는 음식점의 운영시간과 부산의 숙박을 추천해줘"
# # question = "부산의 숙박을 추천해줘"
# question = "부산 해운대의 깨끗한 호텔이랑 내일 부산 날씨, 그리고 서울에서 부산까지 가는 srt 기차표의 제일 이른 시간대를 알려줘"

# messages = [
#             SystemMessage(content=gpt_prompt),
#             HumanMessage(content=question),
#         ]
# decision = structured_llm.invoke(messages)
# rag = decision.rag_queries
# web = decision.web_queries
# weather = decision.weather_queries
# direct = decision.direct
# reason = decision.route_reason
# print("RAG!!!!")
# print(rag)

# print("web!!!!")
# print(web)

# print("weather!!!!")
# print(weather)

# print("direct!!!!")
# print(direct)

# print("reason!!!!")
# print(reason)


In [ ]:
while True:
    full_response = ""
    user_input = input("User: ")
    if user_input.lower() in ["q", "quit"]:
        print("종료합니다.")
        full_response += "종료합니다."
        break
    
    inputs = {"messages": [HumanMessage(content=user_input)]}
    
    result = app.invoke(inputs, config=config)
    
    ai_msg = result["final_answer"]
    route = result.get("route", "알 수 없음")
    summary = result.get("summary", "")

    print(f"AI: {ai_msg}")
    full_response += f"AI: {ai_msg}"
    
    if summary:
        print(f"   └─ [Debug] 📝 요약 발생: {summary}")
        full_response += f"   └─ [Debug] 📝 요약 발생: {summary}"
    
    print("-" * 40)